# Experiment Tracking

This notebook is the central workspace for reviewing benchmark outputs, comparing runs, and keeping the narrative around the project honest.

Authoritative written notes live in `docs/experiment_log.md`. Generated result files live under `data/results/`.

## Usage

1. Run a benchmark through `python -m scripts.run_pipeline ...`
2. Add a short entry to `docs/experiment_log.md`
3. Re-run the cells below to refresh the discovered result files

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = ROOT / "data" / "results"
LOG_PATH = ROOT / "docs" / "experiment_log.md"

ROOT, RESULTS_DIR, LOG_PATH

In [ ]:
def load_json(path: Path):
    with path.open() as f:
        return json.load(f)


def discover_cv_runs(results_dir: Path):
    runs = []
    for path in sorted(results_dir.rglob("cv_results*.json")):
        payload = load_json(path)
        runs.append({
            "path": str(path.relative_to(ROOT)),
            "accuracy_mean": payload.get("accuracy_mean"),
            "accuracy_std": payload.get("accuracy_std"),
            "auc_mean": payload.get("auc_mean"),
            "auc_std": payload.get("auc_std"),
        })
    return runs


runs = discover_cv_runs(RESULTS_DIR)
runs

In [ ]:
for run in sorted(runs, key=lambda item: (item["auc_mean"] or -1), reverse=True):
    print(run["path"])
    print(f"  accuracy = {run['accuracy_mean']:.3f} +/- {run['accuracy_std']:.3f}")
    print(f"  auc      = {run['auc_mean']:.3f} +/- {run['auc_std']:.3f}")
    print()

In [ ]:
print(LOG_PATH.read_text())